# Polyp Segmentation — Kvasir-SEG
**Model:** ResNet50 + MSFF + CBAM | **Loss:** Dice+BCE | **Split:** 70/15/15


In [ ]:
import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU     :", torch.cuda.get_device_name(0))
!pip install -q albumentations==1.3.1
print("Libraries ready!")


## Step 1: Mount Drive & Load Dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
zip_path = "/content/drive/MyDrive/archive.zip"
if not os.path.exists("/content/kvasir-seg"):
    subprocess.run(["unzip", "-q", zip_path, "-d", "/content/"], check=True)
    print("Extracted!")
else:
    print("Already extracted!")
DATA_DIR = "/content/kvasir-seg"
print("Images:", len(os.listdir(os.path.join(DATA_DIR, "images"))))
print("Masks :", len(os.listdir(os.path.join(DATA_DIR, "masks"))))


## Step 2: Imports & Config

In [ ]:
import os, csv, time, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2

CFG = {
    "DATA_DIR"    : "/content/kvasir-seg",
    "IMG_SIZE"    : 352,
    "BATCH_SIZE"  : 8,
    "EPOCHS"      : 100,
    "LR"          : 1e-4,
    "WEIGHT_DECAY": 1e-4,
    "PATIENCE"    : 15,
    "BACKBONE"    : "resnet50",
    "SEED"        : 42,
    "NUM_WORKERS" : 2,
    "OUTPUT_DIR"  : "/content/outputs"
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["OUTPUT_DIR"], exist_ok=True)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(CFG["SEED"])
print("Device  :", DEVICE)
print("DATA_DIR:", CFG["DATA_DIR"])


## Step 3: Dataset & Augmentation (70/15/15)

In [ ]:
def get_train_transforms(sz):
    return A.Compose([
        A.Resize(sz, sz), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.OneOf([A.ElasticTransform(p=0.5), A.GridDistortion(p=0.5), A.OpticalDistortion(p=0.5)], p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])

def get_val_transforms(sz):
    return A.Compose([
        A.Resize(sz, sz),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])

class KvasirSEGDataset(Dataset):
    def __init__(self, imgs, masks, tfm=None):
        self.imgs=imgs; self.masks=masks; self.tfm=tfm
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        img = np.array(Image.open(self.imgs[i]).convert("RGB"))
        msk = np.array(Image.open(self.masks[i]).convert("L"))
        msk = (msk > 127).astype(np.float32)
        if self.tfm:
            a=self.tfm(image=img,mask=msk); img=a["image"]; msk=a["mask"].unsqueeze(0)
        return img, msk

def build_dataloaders(cfg):
    img_dir  = os.path.join(cfg["DATA_DIR"], "images")
    mask_dir = os.path.join(cfg["DATA_DIR"], "masks")
    fn = sorted(os.listdir(img_dir))
    imgs  = [os.path.join(img_dir,  f) for f in fn]
    masks = [os.path.join(mask_dir, f) for f in fn]
    idx = list(range(len(fn))); random.shuffle(idx)
    imgs=[imgs[i] for i in idx]; masks=[masks[i] for i in idx]
    n=len(fn); nt=int(0.70*n); nv=int(0.15*n); sz=cfg["IMG_SIZE"]
    trd=KvasirSEGDataset(imgs[:nt],      masks[:nt],      get_train_transforms(sz))
    vld=KvasirSEGDataset(imgs[nt:nt+nv], masks[nt:nt+nv], get_val_transforms(sz))
    ted=KvasirSEGDataset(imgs[nt+nv:],   masks[nt+nv:],   get_val_transforms(sz))
    print("Train={}  Val={}  Test={}".format(len(trd),len(vld),len(ted)))
    kw=dict(batch_size=cfg["BATCH_SIZE"],num_workers=cfg["NUM_WORKERS"],pin_memory=True)
    return (DataLoader(trd,shuffle=True,**kw), DataLoader(vld,shuffle=False,**kw), DataLoader(ted,shuffle=False,**kw))

train_loader, val_loader, test_loader = build_dataloaders(CFG)

MEAN=np.array([0.485,0.456,0.406]); STD=np.array([0.229,0.224,0.225])
imgs_s,msks_s=next(iter(train_loader))
fig,axes=plt.subplots(2,4,figsize=(14,6))
for i in range(4):
    img=np.clip(imgs_s[i].permute(1,2,0).numpy()*STD+MEAN,0,1)
    axes[0,i].imshow(img); axes[0,i].set_title("Image {}".format(i+1)); axes[0,i].axis("off")
    axes[1,i].imshow(msks_s[i,0],cmap="gray"); axes[1,i].set_title("Mask {}".format(i+1)); axes[1,i].axis("off")
plt.suptitle("Sample Training Batch",fontweight="bold"); plt.tight_layout(); plt.show()


## Step 4: Model (ResNet50 + MSFF + CBAM)

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_c, r=16):
        super().__init__()
        mid = max(in_c // r, 1)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(
            nn.Conv2d(in_c, mid, 1, bias=False), nn.ReLU(True),
            nn.Conv2d(mid, in_c, 1, bias=False))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.fc(self.avg(x)) + self.fc(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        a = x.mean(1, keepdim=True)
        m, _ = x.max(1, keepdim=True)
        return x * self.sig(self.conv(torch.cat([a, m], 1)))

class CBAM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ca = ChannelAttention(c)
        self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))

def cbr(i, o, k=3, p=1):
    return nn.Sequential(
        nn.Conv2d(i, o, k, padding=p, bias=False),
        nn.BatchNorm2d(o), nn.ReLU(True))

class DecoderBlock(nn.Module):
    def __init__(self, in_c, sk_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(cbr(in_c + sk_c, out_c), cbr(out_c, out_c))
        self.attn = CBAM(out_c)
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], 1)
        return self.attn(self.conv(x))

class MSFF(nn.Module):
    def __init__(self, chs, out_c=256):
        super().__init__()
        self.projs = nn.ModuleList([
            nn.Sequential(nn.Conv2d(c, out_c, 1, bias=False),
                          nn.BatchNorm2d(out_c), nn.ReLU(True)) for c in chs])
        self.fuse = cbr(out_c, out_c)
    def forward(self, feats, tgt):
        out = None
        for p, f in zip(self.projs, feats):
            ff  = F.interpolate(p(f), size=tgt, mode="bilinear", align_corners=False)
            out = ff if out is None else out + ff
        return self.fuse(out)

class PolypSegNet(nn.Module):
    def __init__(self, backbone="resnet50", pretrained=True):
        super().__init__()
        b = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        self.enc0 = nn.Sequential(b.conv1, b.bn1, b.relu, b.maxpool)
        self.enc1 = b.layer1
        self.enc2 = b.layer2
        self.enc3 = b.layer3
        self.enc4 = b.layer4
        ec = [64, 256, 512, 1024, 2048]
        self.msff = MSFF([ec[2], ec[3], ec[4]], 256)
        self.dec4 = DecoderBlock(256, ec[1], 128)
        self.dec3 = DecoderBlock(128, ec[0],  64)
        self.dec2 = DecoderBlock(64,  0,       32)
        self.dec1 = DecoderBlock(32,  0,       16)
        self.head = nn.Conv2d(16, 1, 1)
    def forward(self, x):
        H, W = x.shape[2:]
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        neck = self.msff([e2, e3, e4], (e2.shape[2], e2.shape[3]))
        d = self.dec4(neck, e1)
        d = self.dec3(d,    e0)
        d = self.dec2(d,    None)
        d = self.dec1(d,    None)
        d = F.interpolate(d, (H, W), mode="bilinear", align_corners=False)
        return torch.sigmoid(self.head(d))

model = PolypSegNet(backbone=CFG["BACKBONE"]).to(DEVICE)
n_p = sum(p.numel() for p in model.parameters()) / 1e6
print("Backbone : {}".format(CFG["BACKBONE"]))
print("Params   : {:.2f}M".format(n_p))
with torch.no_grad():
    _out = model(torch.randn(1,3,352,352).to(DEVICE))
print("Output   : {}  OK!".format(tuple(_out.shape)))



## Step 5: Loss & Metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0): super().__init__(); self.s=smooth
    def forward(self, p, t):
        p=p.contiguous().view(-1); t=t.contiguous().view(-1)
        inter=(p*t).sum()
        return 1-(2*inter+self.s)/(p.sum()+t.sum()+self.s)

class DiceBCELoss(nn.Module):
    def __init__(self,alpha=0.5):
        super().__init__(); self.alpha=alpha; self.dice=DiceLoss(); self.bce=nn.BCELoss()
    def forward(self,p,t): return self.alpha*self.dice(p,t)+(1-self.alpha)*self.bce(p,t)

class MetricMeter:
    def __init__(self,thr=0.5): self.thr=thr; self.reset()
    def reset(self): self.s={"dice":0.,"iou":0.,"prec":0.,"rec":0.,"f2":0.,"mae":0.}; self.n=0
    def update(self,pred,tgt):
        pred=pred.detach().cpu(); tgt=tgt.detach().cpu(); pb=(pred>self.thr).float()
        for b in range(pred.shape[0]):
            p=pb[b].view(-1); t=tgt[b].view(-1); sm=1e-6
            TP=(p*t).sum(); FP=(p*(1-t)).sum(); FN=((1-p)*t).sum()
            self.s["dice"]+=((2*TP+sm)/(2*TP+FP+FN+sm)).item()
            self.s["iou"] +=((TP+sm)/(TP+FP+FN+sm)).item()
            self.s["prec"]+=((TP+sm)/(TP+FP+sm)).item()
            self.s["rec"] +=((TP+sm)/(TP+FN+sm)).item()
            self.s["f2"]  +=((5*TP+sm)/(5*TP+4*FN+FP+sm)).item()
            self.s["mae"] +=(pred[b]-tgt[b]).abs().mean().item()
            self.n+=1
    def compute(self): return {k:v/self.n for k,v in self.s.items()} if self.n else self.s

print("Loss & Metrics ready!")


## Step 6: Training (Early Stopping + LR Scheduler + Best Weight)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=15, path="best_model.pth", delta=1e-4):
        self.patience=patience; self.path=path; self.delta=delta
        self.best=None; self.counter=0; self.triggered=False
    def __call__(self, score, model):
        if self.best is None or score > self.best + self.delta:
            self.best=score; self.counter=0
            torch.save(model.state_dict(), self.path)
            print("  Best model saved  (val_dice={:.4f})".format(score))
        else:
            self.counter += 1
            print("  EarlyStopping: {}/{}".format(self.counter, self.patience))
            if self.counter >= self.patience: self.triggered = True

criterion = DiceBCELoss(alpha=0.5)
optimizer = optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
scheduler = CosineAnnealingLR(optimizer, T_max=CFG["EPOCHS"], eta_min=1e-6)
ckpt_path = os.path.join(CFG["OUTPUT_DIR"], "best_model.pth")
stopper   = EarlyStopping(patience=CFG["PATIENCE"], path=ckpt_path)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total = 0.; meter = MetricMeter()
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, msks in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            msks = msks.to(DEVICE, non_blocking=True)
            if train: optimizer.zero_grad()
            preds = model(imgs)
            loss  = criterion(preds, msks)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item() * imgs.size(0)
            meter.update(preds, msks)
    m = meter.compute()
    m["loss"] = total / len(loader.dataset)
    return m

history = {"train_loss":[], "train_dice":[], "val_loss":[], "val_dice":[], "val_iou":[]}
print("="*55)
print(" TRAINING START")
print("="*55)

for epoch in range(1, CFG["EPOCHS"]+1):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    vl = run_epoch(val_loader,   train=False)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]
    history["train_loss"].append(tr["loss"])
    history["train_dice"].append(tr["dice"])
    history["val_loss"].append(vl["loss"])
    history["val_dice"].append(vl["dice"])
    history["val_iou"].append(vl["iou"])
    print("Epoch [{:03d}/{}] {:.0f}s  lr={:.2e}".format(epoch, CFG["EPOCHS"], time.time()-t0, lr))
    print("  Train  loss={:.4f}  dice={:.4f}".format(tr["loss"], tr["dice"]))
    print("  Val    loss={:.4f}  dice={:.4f}  iou={:.4f}".format(vl["loss"], vl["dice"], vl["iou"]))
    stopper(vl["dice"], model)
    if stopper.triggered:
        print("EarlyStopping! Best val Dice={:.4f}".format(stopper.best))
        break



## Step 7: Training Curves

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(15,4))
e=range(1,len(history["train_loss"])+1)
ax[0].plot(e,history["train_loss"],label="Train",color="steelblue")
ax[0].plot(e,history["val_loss"],  label="Val",  color="tomato")
ax[0].set_title("Loss (Dice+BCE)",fontweight="bold"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(e,history["train_dice"],label="Train",color="steelblue")
ax[1].plot(e,history["val_dice"],  label="Val",  color="tomato")
ax[1].set_title("Dice Coefficient",fontweight="bold"); ax[1].legend(); ax[1].grid(alpha=0.3)
ax[2].plot(e,history["val_iou"],color="seagreen")
ax[2].set_title("Validation IoU",fontweight="bold"); ax[2].grid(alpha=0.3)
plt.suptitle("Training History",fontsize=13,fontweight="bold"); plt.tight_layout()
out_c=os.path.join(CFG["OUTPUT_DIR"],"training_curves.png")
plt.savefig(out_c,dpi=150,bbox_inches="tight"); plt.show(); print("Saved:",out_c)


## Step 8: Test Evaluation

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
tm = run_epoch(test_loader, train=False)
print("="*40)
print(" TEST RESULTS (Best Weights)")
print("="*40)
for k,v in tm.items(): print("  {:12s}: {:.4f}".format(k,v))
print("="*40)
rp=os.path.join(CFG["OUTPUT_DIR"],"test_results.csv")
with open(rp,"w",newline="") as f:
    w=csv.writer(f); w.writerow(["Metric","Value"])
    for k,v in tm.items(): w.writerow([k,"{:.6f}".format(v)])
print("Saved:",rp)


## Step 9: Qualitative Results + Grad-CAM

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model=model; self._f=None; self._g=None
        layer.register_forward_hook(lambda _,__,o: setattr(self,"_f",o.detach()))
        layer.register_full_backward_hook(lambda _,__,g: setattr(self,"_g",g[0].detach()))
    def generate(self, x):
        self.model.eval(); x=x.clone().requires_grad_(True)
        self.model(x).mean().backward()
        w=self._g.mean((2,3),keepdim=True)
        cam=F.relu((w*self._f).sum(1,keepdim=True)).squeeze().cpu().numpy()
        return (cam-cam.min())/(cam.max()-cam.min()+1e-8)

gcam=GradCAM(model, model.enc4[-1].conv3)
model.eval()
imgs_v,msks_v=next(iter(test_loader))
imgs_v=imgs_v[:4].to(DEVICE); msks_v=msks_v[:4]
with torch.no_grad(): preds_v=model(imgs_v).cpu()

fig,axes=plt.subplots(4,5,figsize=(18,14))
for j,t in enumerate(["Original","Ground Truth","Prediction","Grad-CAM","Error Map"]):
    axes[0,j].set_title(t,fontweight="bold",fontsize=10)

for i in range(4):
    orig=np.clip(imgs_v[i].cpu().permute(1,2,0).numpy()*STD+MEAN,0,1)
    orig_u8=(orig*255).astype(np.uint8)
    with torch.enable_grad(): cam=gcam.generate(imgs_v[i:i+1])
    cam=cv2.resize(cam,(CFG["IMG_SIZE"],CFG["IMG_SIZE"]))
    hm=cv2.applyColorMap((cam*255).astype(np.uint8),cv2.COLORMAP_JET)
    hm=cv2.cvtColor(hm,cv2.COLOR_BGR2RGB)
    cam_ov=np.clip(0.5*hm+0.5*orig_u8,0,255).astype(np.uint8)
    pred_np=preds_v[i,0].numpy(); gt_np=msks_v[i,0].numpy()
    axes[i,0].imshow(orig); axes[i,1].imshow(gt_np,cmap="gray")
    axes[i,2].imshow(pred_np>0.5,cmap="gray"); axes[i,3].imshow(cam_ov)
    axes[i,4].imshow(np.abs(pred_np-gt_np),cmap="hot",vmin=0,vmax=1)
    for j in range(5): axes[i,j].axis("off")

plt.suptitle("Qualitative Results",fontsize=13,fontweight="bold"); plt.tight_layout()
out_q=os.path.join(CFG["OUTPUT_DIR"],"qualitative_results.png")
plt.savefig(out_q,dpi=150,bbox_inches="tight"); plt.show(); print("Saved:",out_q)


## Step 10: Download All Results

In [ ]:
from google.colab import files
print("Downloading results...")
for fname in ["best_model.pth","training_curves.png","qualitative_results.png","test_results.csv"]:
    p=os.path.join(CFG["OUTPUT_DIR"],fname)
    if os.path.exists(p): files.download(p); print("  Downloaded:",fname)
    else: print("  Not found :",fname)
